# Notebook 2: Baseline Models (RB and SB)
**Author:** Dr. Mo Medwani

This notebook implements the two baseline models:
- **Rule-Based (RB):** Inspired by the Buckwalter Arabic Morphological Analyzer (BAMA)
- **Statistical (SB):** Inspired by the MADA (Morphological Analysis and Disambiguation for Arabic) system

These baselines establish the performance floor against which all neural models are compared.

## Setup

In [ ]:
import re
import unicodedata
from collections import defaultdict
import numpy as np
print("Imports loaded.")

## 1. Rule-Based (RB) Baseline
**Architecture:** Lexicon + Compatibility Tables + Disambiguation Rules  
**MSA WLA:** 82.1% (Table 12)  
**Key Strength:** No training data required  
**Key Weakness:** Cannot handle unseen words or dialectal variation

In [ ]:
class RuleBasedAnalyzer:
    """
    Rule-Based morphological analyzer inspired by BAMA (Buckwalter, 2004).
    Implements a simplified version of the prefix-stem-suffix compatibility paradigm.
    Separate configurations for MSA and each DA variety share a common core.
    """
    def __init__(self, dialect='MSA'):
        self.dialect = dialect
        # Core lexicon: stem -> (POS, features)
        self.stem_lexicon = {
            'كتب': ('VERB', {'root': 'ك-ت-ب', 'pattern': 'فعل', 'voice': 'active'}),
            'كتاب': ('NOUN', {'root': 'ك-ت-ب', 'pattern': 'فعال', 'gender': 'masc'}),
            'مدرسة': ('NOUN', {'root': 'د-ر-س', 'pattern': 'مفعلة', 'gender': 'fem'}),
            'ذهب': ('VERB', {'root': 'ذ-ه-ب', 'pattern': 'فعل', 'voice': 'active'}),
            'بيت': ('NOUN', {'root': 'ب-ي-ت', 'pattern': 'فعل', 'gender': 'masc'}),
        }
        # Prefix compatibility table
        self.prefixes = {
            'MSA': {'ال': 'DET', 'و': 'CONJ', 'ب': 'PREP', 'ل': 'PREP', 'ك': 'PREP'},
            'EGY': {'ال': 'DET', 'و': 'CONJ', 'ب': 'PREP', 'ف': 'CONJ', 'م': 'NEG'},
        }
        # Suffix compatibility table
        self.suffixes = {
            'ة': ('NOUN', {'gender': 'fem'}),
            'ات': ('NOUN', {'number': 'plural', 'gender': 'fem'}),
            'ون': ('NOUN', {'number': 'plural', 'gender': 'masc'}),
            'ت': ('VERB', {'person': '3rd', 'gender': 'fem', 'tense': 'past'}),
            'ي': ('PRON', {'person': '1st'}),
        }
        print(f"Rule-Based Analyzer initialized for dialect: {dialect}")

    def segment(self, word):
        """Attempt to segment word into prefix + stem + suffix."""
        prefix_table = self.prefixes.get(self.dialect, self.prefixes['MSA'])
        for prefix in sorted(prefix_table.keys(), key=len, reverse=True):
            if word.startswith(prefix) and len(word) > len(prefix) + 1:
                stem = word[len(prefix):]
                for suffix, (pos, feats) in self.suffixes.items():
                    if stem.endswith(suffix) and len(stem) > len(suffix):
                        core = stem[:-len(suffix)]
                        return {'prefix': prefix, 'stem': core, 'suffix': suffix}
                return {'prefix': prefix, 'stem': stem, 'suffix': ''}
        return {'prefix': '', 'stem': word, 'suffix': ''}

    def analyze(self, word):
        """Full morphological analysis pipeline."""
        segments = self.segment(word)
        stem = segments['stem']
        # Lookup stem in lexicon
        if stem in self.stem_lexicon:
            pos, features = self.stem_lexicon[stem]
            return {'word': word, 'segments': segments, 'POS': pos, 'features': features, 'confidence': 'HIGH'}
        else:
            return {'word': word, 'segments': segments, 'POS': 'UNK', 'features': {}, 'confidence': 'LOW'}

    def analyze_sentence(self, sentence):
        """Analyze all words in a sentence."""
        words = sentence.split()
        return [self.analyze(word) for word in words]

# --- Demonstration ---
rb_msa = RuleBasedAnalyzer(dialect='MSA')
sentence = "الكتاب مكتوب باللغة العربية"
analysis = rb_msa.analyze_sentence(sentence)
for item in analysis:
    print(f"Word: {item['word']:15} | POS: {item['POS']:6} | Segments: {item['segments']} | Confidence: {item['confidence']}")

## 2. Statistical (SB) Baseline
**Architecture:** Rule-based candidate generation + Maximum Entropy disambiguation  
**MSA WLA:** 85.4% (Table 12)  
**Key Strength:** Handles ambiguity better than pure rule-based  
**Key Weakness:** Performance plateaus at ~500K training words

In [ ]:
class StatisticalAnalyzer:
    """
    Statistical morphological analyzer inspired by MADA (Roth et al., 2008).
    Uses rule-based candidate generation followed by statistical disambiguation.
    Features: character n-grams, contextual words, POS context.
    """
    def __init__(self, dialect='MSA'):
        self.dialect = dialect
        self.rb_analyzer = RuleBasedAnalyzer(dialect)
        # Simulated learned weights (in production, these come from training on annotated data)
        self.pos_priors = {
            'NOUN': 0.35, 'VERB': 0.25, 'ADJ': 0.12, 'PREP': 0.10,
            'CONJ': 0.08, 'PRON': 0.05, 'DET': 0.03, 'OTHER': 0.02
        }
        # Simulated contextual transition probabilities (bigram POS model)
        self.pos_transitions = {
            ('DET', 'NOUN'): 0.90, ('NOUN', 'VERB'): 0.45, ('VERB', 'NOUN'): 0.40,
            ('PREP', 'DET'): 0.70, ('PREP', 'NOUN'): 0.25, ('CONJ', 'NOUN'): 0.35,
        }
        print(f"Statistical Analyzer initialized for dialect: {dialect}")

    def _extract_char_ngrams(self, word, n=3):
        """Extract character n-grams as features."""
        return [word[i:i+n] for i in range(len(word) - n + 1)]

    def _generate_candidates(self, word):
        """Use rule-based system to generate candidate analyses."""
        rb_result = self.rb_analyzer.analyze(word)
        # In a real system, this would generate multiple possible analyses
        candidates = [rb_result]
        # Add alternative analyses based on common patterns
        if word.endswith('ة'):
            candidates.append({'word': word, 'POS': 'NOUN', 'features': {'gender': 'fem'}, 'confidence': 'MEDIUM'})
        return candidates

    def _score_candidate(self, candidate, context_pos=None):
        """Score a candidate analysis using learned weights."""
        pos = candidate['POS']
        score = self.pos_priors.get(pos, 0.01)
        # Apply contextual boost
        if context_pos and (context_pos, pos) in self.pos_transitions:
            score *= (1 + self.pos_transitions[(context_pos, pos)])
        return score

    def disambiguate(self, word, context_pos=None):
        """Select the best analysis from candidates."""
        candidates = self._generate_candidates(word)
        scored = [(c, self._score_candidate(c, context_pos)) for c in candidates]
        best = max(scored, key=lambda x: x[1])
        return best[0]

    def analyze_sentence(self, sentence):
        """Analyze sentence with contextual disambiguation."""
        words = sentence.split()
        results = []
        prev_pos = None
        for word in words:
            analysis = self.disambiguate(word, context_pos=prev_pos)
            results.append(analysis)
            prev_pos = analysis['POS']
        return results

# --- Demonstration ---
sb_msa = StatisticalAnalyzer(dialect='MSA')
sentence = "الكتاب مكتوب باللغة العربية"
analysis = sb_msa.analyze_sentence(sentence)
for item in analysis:
    print(f"Word: {item['word']:15} | POS: {item['POS']:6} | Confidence: {item['confidence']}")

## Performance Comparison: RB vs SB

In [ ]:
import pandas as pd

# Results from Dissertation Table 12 (MSA WLA)
comparison = {
    'Model': ['Rule-Based (RB)', 'Statistical (SB)'],
    'MSA WLA (%)': [82.1, 85.4],
    'Training Data Required': ['None', 'Annotated corpus'],
    'Handles Unseen Words': ['No', 'Partially'],
    'Dialectal Support': ['Limited', 'With retraining'],
    'Speed (words/sec)': ['~50,000', '~5,000']
}
df = pd.DataFrame(comparison)
print(df.to_string(index=False))